# 01. Attribute lookup

**Tier:** Deep. **Why you'd use this:** You read `obj.method` and need to know which class supplied it, how the MRO orders parents, and what `__slots__` removes from the usual `__dict__`.

**After this lesson you can:** say which class supplies an attribute; read an MRO; tell `__getattr__` from `__getattribute__`; see what `__slots__` removes.
About 35 minutes. **Uses:** [Classes](../../course/11-classes/01-classes-lesson.ipynb), [Inheritance](../../course/11-classes/03-inheritance-lesson.ipynb), [The iteration protocol](../07-iteration/01-protocol-lesson.ipynb)


## Quick review

1. What do `iter(x)` and `next(it)` do?
2. Why is a second `for` over the same generator empty?

<details><summary>Answers</summary>

`iter` gets an iterator; `next` pulls one value. The generator is already spent.
</details>


## The idea

`obj.name` is a search. Python looks in the instance `__dict__` first, then on the class, then along the class's method resolution order. The first hit wins. A function found on the class is turned into a bound method on the way out, so `obj.method` already knows `obj`.

`super()` continues that search after the class that is currently running, following the same MRO. It is not "the parent". In a diamond, the order is what keeps a shared base from running twice.

<details><summary>Explain more</summary>

A descriptor is an object on the class with `__get__`. Functions and `property` are descriptors. `__getattribute__` runs on every lookup. `__getattr__` runs only when the normal search fails.

`__slots__` declares the instance's attributes in the object itself and omits `__dict__`. You cannot assign a name that was not listed.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

The MRO is the linearization of multiple inheritance. `super()` is the next class in that list, not a named parent. `__slots__` is closer to a fixed set of fields than to a Java class, which never had a per-instance dictionary.
</details>


## See it work

A diamond of roles should run each class's greeting once, in a stable order. `super()` follows the MRO of the actual instance, so `D` runs `B`, then `C`, then `A`, then `object`. Hard-coding `Base.ping(self)` would skip `C` or call `A` twice, depending on which class you named.


In [1]:
class A:
    def ping(self) -> str:
        return "A"


class B(A):
    def ping(self) -> str:
        return "B " + super().ping()


class C(A):
    def ping(self) -> str:
        return "C " + super().ping()


class D(B, C):
    def ping(self) -> str:
        return "D " + super().ping()


print([cls.__name__ for cls in D.__mro__])
print(D().ping())

['D', 'B', 'C', 'A', 'object']
D B C A


| class of `self` | `super().ping` continues at |
| --- | --- |
| D | B |
| B | C |
| C | A |
| A | object, and `A.ping` returns |

```
D -> B -> C -> A -> object
D().ping() == "D B C A"
```

`C` is not `B`'s parent. It is the next class in `D`'s MRO when `B.ping` asks for `super()`. The same `B.ping`, on a plain `B()` instance, would continue at `A`, because that instance's MRO is `B, A, object`.


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `child.ping()` see `Child.label` when `ping` is defined on `Base`?
2. Why does `B.ping` call `C.ping` for a `D` instance and not for a `B` instance?
3. Why doesn't `Loose().other` need `__getattribute__`?

<details><summary>Answers</summary>

1. `self` is the `Child` instance. `self.label` starts the search at that instance, finds `Child.label`, and stops. The function's class does not freeze the attribute search.
2. `super()` uses the MRO of `type(self)`, starting after `B`. For a `D`, the next class is `C`. For a `B`, the next class is `A`. The same function, two MROs.
3. `other` is missing, so the normal search raises `AttributeError` and only then calls `__getattr__`. `__getattribute__` already ran the search. You override it only when you want to intercept names that do exist.
</details>


In [2]:
class Base:
    label = "base"

    def ping(self) -> str:
        return self.label


class Child(Base):
    label = "child"


child = Child()
child.note = "instance"
print(child.label)
print(child.ping())
print(child.note)
print(child.ping.__self__ is child)
print(child.ping is child.ping)

child
child
instance
True
False


<details><summary>Check your answers</summary>

- `child.label` is `child`. The instance dict has no `label`, so lookup finds `Child.label` and stops before `Base.label`.
- `child.ping()` is also `child`. The function lives on `Base`, and `self.label` runs the search again from the instance.
- `note` is on the instance. It never reaches the class.
- `ping.__self__ is child` is true. Looking up the function produced a bound method whose instance is `child`.
- `child.ping is child.ping` is false. Each lookup builds a new bound method. The function on the class is shared. The bound method is not.
</details>


## Why it works this way

The instance `__dict__` is a per-object mapping from names to objects. Writing `child.note = "instance"` stores a pointer there. Reading walks that dict before anything else, which is why an instance attribute shadows a class attribute of the same name. The class attribute is still on the class. Other instances that never assigned the name still find it.

After the instance dict misses, lookup uses the class of the instance, then each class in the MRO. The MRO is a single list computed when the class is created, from the bases, so every lookup agrees on the order. The algorithm merges the parents' MROs so that a class stays ahead of its own bases and the base lists stay in order. In the diamond, that merge produces `D, B, C, A, object`. `A` appears once. `super()` does not mean "my first base". It means "continue this MRO after the class whose method is running". That is why `B.ping` reaches `C` when `self` is a `D`, and reaches `A` when `self` is a `B`. Cooperative `super()` is what makes the diamond call each class once. Naming a parent calls whoever you named, and the other branch disappears.

A function on the class is a descriptor. Its `__get__` builds a bound method that holds the instance. The function object on the class is unchanged. The bound method is a new object per lookup, which is why `child.ping is child.ping` is false even though both are the same function bound to the same instance. `property` is the same protocol with your code in `__get__`. Reading the name calls the getter. The instance dict is not consulted for that name when a data descriptor sits on the class, which is why a property stays in charge even if you try to stash a value under the same name.

`__getattribute__` is the method that performs this whole search. Override it and you replace the search. `object.__getattribute__` is how you run the normal one from inside the override, which you must do or the lookup recurses forever. `__getattr__` is different. It is called only after the normal search raises `AttributeError`. A missing name can be invented there. A name that exists never reaches `__getattr__`.

`__slots__` replaces the instance dict with fixed slots in the object layout. Lookup of a slotted name reads that field. There is no `__dict__`, so an arbitrary new name has nowhere to go and raises `AttributeError`. The object is smaller by about the size of a dict header, which matters when you have millions of them, and it does not matter when you have dozens. `sys.getsizeof` on one instance is a CPython detail and depends on the build, so that cell is volatile. The stable fact is `hasattr(obj, "__dict__")`.

The design reason for the search is that methods live on the class once, not copied onto every instance, and subclasses can override a name without editing the base. The design reason for descriptors is that attribute access can run code, so `property`, methods, and slots share one hook. The cost is that a lookup is a walk, not a single machine load, unless the attribute is a slot.


In [3]:
class Reading:
    def __init__(self) -> None:
        self._n = 1

    @property
    def n(self) -> int:
        return self._n


reading = Reading()
print(reading.n)
print("n" in reading.__dict__)

1
False


In [4]:
class Loose:
    def __getattr__(self, name: str) -> str:
        return "missing " + name


class Forced:
    def __getattr__(self, name: str) -> str:
        return "missing " + name

    def __getattribute__(self, name: str):
        if name == "hidden":
            return "forced"
        return object.__getattribute__(self, name)


print(Loose().other)
print(Forced().hidden, Forced().other)

missing other
forced missing other


In [5]:
import sys


class Slotted:
    __slots__ = ("x",)

    def __init__(self, x: int) -> None:
        self.x = x


class Plain:
    def __init__(self, x: int) -> None:
        self.x = x


slotted = Slotted(1)
plain = Plain(1)
print(hasattr(slotted, "__dict__"), hasattr(plain, "__dict__"))
print(sys.getsizeof(slotted), sys.getsizeof(plain))

False True
40 48


## When to use it

Put shared behavior and shared constants on the class. Put per-object data on the instance. Use `super()` in every override that should cooperate, and do not name a parent class unless you mean to skip the MRO. Use a `property` when reading or writing a name must run code. Use a plain attribute when it is just data.

Use `__getattr__` for a genuine fallback, such as a proxy that forwards unknown names. Do not use it to hide typos. An override of `__getattribute__` is rare. If you write one, call `object.__getattribute__` for the names you do not handle.

Use `__slots__` when you have a very large number of small objects and you have measured the dict overhead. Do not use it as a default. It blocks arbitrary attributes, complicates multiple inheritance, and saves little on a handful of instances. A normal class with `__dict__` is the right tool until a profile says otherwise.


## Common mistake

Calling `A.ping(self)` from both branches of a diamond runs `A` twice and skips the sibling. The string looks plausible. Nothing raises. The MRO was ignored.


In [6]:
class DirectA:
    def ping(self) -> str:
        return "A"


class DirectB(DirectA):
    def ping(self) -> str:
        return "B " + DirectA.ping(self)


class DirectC(DirectA):
    def ping(self) -> str:
        return "C " + DirectA.ping(self)


class DirectD(DirectB, DirectC):
    def ping(self) -> str:
        return "D " + DirectB.ping(self)


print(DirectD().ping())

D B A


The print is `D B A`. `C` never ran, and the call named `A` directly from `B`. The fix uses `super()` in each class, which follows `D`'s MRO through `C` before `A`.


In [7]:
print(D().ping())

D B C A


## Try it

Add `label = "base"` on `Base` and delete `label` from `Child`. What does `Child().ping()` return, and which step of the search changed?


## What this explains

- [Classes](../../course/11-classes/01-classes-lesson.ipynb): methods are functions on the class. Lookup binds them to the instance.
- [Inheritance](../../course/11-classes/03-inheritance-lesson.ipynb): `super()` follows the MRO. The diamond here is that rule with two bases.
- [Properties](../../course/11-classes/04-properties-lesson.ipynb): a property is a descriptor. Reading the name runs `__get__` instead of fetching a value from the instance dict.


## Recap

- Lookup is the instance dict, then the class, then the MRO. The first hit wins.
- `super()` continues that MRO. It is not a named parent. Functions and properties are descriptors.
- `__getattr__` is the fallback. `__slots__` drops `__dict__` for a fixed layout.

Next: [The import system](../09-imports/01-import-system-lesson.ipynb).
